# Simulación de portafolio con Cholesky
María Fernanda Gómez Flores

Simulación de AAPL, MSFT y NVDA a un año con 10,000 escenarios.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

### Precios y rendimientos
Descargamos los precios de las acciones y sacamos los rendimientos logarítmicos diarios.

In [ ]:
tickers = ["AAPL", "MSFT", "NVDA"]
prices = yf.download(tickers=tickers, start="2023-11-01", end="2026-11-01")["Close"]
prices = prices[tickers].dropna()
rets = np.log(prices / prices.shift(1)).dropna()
rets

### Correlación
Calculamos la correlación entre los rendimientos de las tres acciones.

In [ ]:
rets.corr()

### Covarianza
Sacamos la matriz de covarianzas de los rendimientos diarios.

In [ ]:
Sigma = rets.cov()
Sigma

### Cholesky
Descomponemos la matriz de covarianzas para poder simular los rendimientos tomando en cuenta la relación entre las acciones.

In [ ]:
L = np.linalg.cholesky(Sigma)
L

### Rendimiento promedio
Calculamos el rendimiento promedio diario y lo pasamos a anual usando 252 días.

In [ ]:
mu = rets.mean().values * 252
mu

### Simulación
Generamos 10,000 números aleatorios normales para cada acción.

In [ ]:
M = 10000
rng = np.random.default_rng(42)
epsilon = rng.standard_normal((len(tickers), M))
epsilon

### Rendimientos correlacionados
Multiplicamos los números aleatorios por la matriz de Cholesky para obtener los movimientos correlacionados.

In [ ]:
Z = L @ epsilon
Z.T

### Precios futuros
Simulamos los precios de cada acción dentro de un año. Aquí ya estamos usando la volatilidad que viene de la matriz de covarianzas.

In [ ]:
T = 1
n_dias = 252
S0 = prices.iloc[-1].values
mu = rets.mean().values * n_dias
vol = rets.std().values * np.sqrt(n_dias)

ST = S0 * np.exp(
    (mu - 0.5 * vol**2) * T
    + np.sqrt(n_dias * T) * Z.T
)
ST = pd.DataFrame(ST, columns=tickers)
ST.head()

### Valor del portafolio
Tomamos un capital inicial de $100,000 y lo dividimos en partes iguales entre las tres acciones. Con eso calculamos cuántas acciones compramos y cuánto valdría el portafolio en cada escenario.

In [ ]:
V0 = 100000
w = np.array([1/3, 1/3, 1/3])
acciones = V0 * w / S0
VT = ST.to_numpy() @ acciones

print(f"Capital inicial: ${V0:,.2f}")
print(f"Valor futuro promedio: ${VT.mean():,.2f}")

### Gráfica del valor futuro
Primero vemos cómo se distribuyen los valores futuros del portafolio sin restarle todavía los $100,000 iniciales.

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(VT, bins=50, edgecolor="black")
plt.axvline(V0, color="red", linestyle="--", label="Capital inicial")
plt.title("Distribución del valor futuro del portafolio")
plt.xlabel("Valor futuro ($)")
plt.ylabel("Frecuencia")
plt.legend()
plt.show()

### Ganancias y pérdidas
Al valor futuro de cada simulación le restamos el capital inicial.

$$P\&L = V_T - V_0$$

Si sale negativo es pérdida y si sale positivo es ganancia.

In [ ]:
PnL = VT - V0
print(f"Ganancia o pérdida promedio: ${PnL.mean():,.2f}")
print(f"Probabilidad simulada de pérdida: {(PnL < 0).mean():.2%}")
PnL

### Gráfica de ganancias y pérdidas
Ahora graficamos los resultados después de restar el capital inicial. El cero indica que no ganamos ni perdimos.

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(PnL, bins=50, edgecolor="black")
plt.axvline(0, color="red", linestyle="--", label="Ganancia/Pérdida = 0")
plt.title("Distribución de ganancias y pérdidas del portafolio")
plt.xlabel("Ganancia / Pérdida ($)")
plt.ylabel("Frecuencia")
plt.legend()
plt.show()